# 🎬 OpusClip Pipeline — Google Colab Demo

**OpusClip** is an AI-powered pipeline that transforms long-form videos (YouTube or local)
into short, publication-ready vertical clips with:
- ✂️ **Smart Clip Selection** — LLM-based virality scoring
- 🎯 **Face-Aware Cropping** — MediaPipe + SmartDirector crop logic
- 🎨 **Bilingual Subtitles** — ASS-format karaoke for Arabic & English
- 🚀 **GPU-Accelerated** — CUDA Whisper + NVENC encoding (when available)

---
### ⚙️ Runtime Setup
Before running any cells:
1. **Runtime → Change runtime type → T4 GPU** (recommended)
2. Set your **OPUSCLIP_API_KEY** in the Secrets panel (🔑) or use the `os.environ` cell below

---
## 1. Install Dependencies

Installs the `opusclip` package and required system binaries (ffmpeg, fonts).
This cell typically runs in **2–3 minutes** on Colab.

In [ ]:
import subprocess, sys, os, pathlib, json

def _sh(cmd, label="", check=False):
    print(f"  → {label or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr.strip():
        print(f"    ⚠ {r.stderr.strip()[:200]}")
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return r

print("📦 Installing opusclip, faster-whisper, opencv, dlib …")
_sh("pip install -q opusclip faster-whisper opencv-python dlib", "opusclip + deps")

print("🔤 Installing fonts …")
_sh("apt-get install -y -q fonts-noto-core fonts-noto-extra fonts-dejavu-core > /dev/null 2>&1",
    "Noto + DejaVu fonts")

print("🎥 Checking ffmpeg …")
_sh("ffmpeg -version", "ffmpeg version")

---
## 2. Configuration

Set your **API key** and choose a video source.

> **Free API options:**
> - [opencode.ai](https://opencode.ai/zen) — `deepseek-v4-flash-free` (default)
> - [Groq](https://console.groq.com) — `llama-3.3-70b-versatile`
> - [Gemini](https://aistudio.google.com/apikey) — `gemini-2.0-flash`

You can store your API key in Colab's **Secrets** (🔑) panel as `OPUSCLIP_API_KEY`,
or set it directly in the cell below.

In [ ]:
# ── API Key ──────────────────────────────────────────────────
# Option A: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ["OPUSCLIP_API_KEY"] = userdata.get("OPUSCLIP_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except (ImportError, ValueError, KeyError):
    # Option B: Set your key directly here
    os.environ["OPUSCLIP_API_KEY"] = ""    # ← paste your key here
    print("⚠  No Colab Secrets found — using inline key (if set)")

# ── LLM Configuration (optional — defaults shown) ────────────
os.environ["LLM_BASE_URL"] = "https://opencode.ai/zen/v1"      # Groq, Gemini, etc.
os.environ["LLM_MODEL"]    = "deepseek-v4-flash-free"          # llama-3.3-70b-versatile, gemini-2.0-flash, …
# os.environ["WHISPER_MODEL"] = "large-v3"                     # or "medium", "small"

# ── Verify ──────────────────────────────────────────────────
assert os.environ.get("OPUSCLIP_API_KEY"), "❌ OPUSCLIP_API_KEY is not set!"
print(f"✅  LLM     : {os.environ.get('LLM_BASE_URL')} / {os.environ.get('LLM_MODEL')}")
print(f"✅  Key     : …{os.environ['OPUSCLIP_API_KEY'][-4:]} (last 4 chars)")

---
## 3. Choose Your Video Source

Set `INPUT_MODE` to `"youtube"` (paste a URL) or `"upload"` (pick a local file).

In [ ]:
# ── Pick ONE mode ────────────────────────────────────────────
INPUT_MODE  = "youtube"   # "upload"  |  "youtube"
YOUTUBE_URL = "https://youtu.be/OwDU3GH9cGI?si=MeHrfxVtdIUK5u9c"  # paste URL if "youtube"

# ── Resolve video path ──────────────────────────────────────
VIDEO_PATH = None
if INPUT_MODE == "upload":
    from google.colab import files as _cf
    print("📂  Select your video file …")
    _uploaded = _cf.upload()
    VIDEO_PATH = list(_uploaded.keys())[0]
elif INPUT_MODE == "youtube":
    assert YOUTUBE_URL, "Set YOUTUBE_URL above!"
    out_dir = pathlib.Path("opusclip_output")
    out_dir.mkdir(exist_ok=True)
    VIDEO_PATH = str(out_dir / "source_video.mp4")
    if not os.path.exists(VIDEO_PATH):
        print(f"⬇️   Downloading: {YOUTUBE_URL}")
        _sh(
            f'yt-dlp --format "bestvideo[height<=1080][ext=mp4]'
            f'+bestaudio[ext=m4a]/best[height<=1080]" '
            f'--merge-output-format mp4 --no-playlist '
            f'--output "{VIDEO_PATH}" "{YOUTUBE_URL}"',
            "yt-dlp download"
        )
    else:
        print(f"✓  Cached: {VIDEO_PATH}")

print(f"\n🎬  Source: {VIDEO_PATH}")

---
## 4. Run Pipeline

This cell uses `opusclip`'s production ``ProviderFactory`` and ``Pipeline`` classes.
All 10 pipeline steps run automatically:

| # | Step | Description |
|---|------|-------------|
| 1 | Validating input | Checks source URL or file path |
| 2 | Reading metadata | Extracts video info via ffprobe |
| 3 | Transcribing audio | faster-whisper (CUDA if available) |
| 4 | Repairing transcript | Fills missing words from Whisper |
| 5 | Selecting clips | LLM-based virality scoring |
| 6 | Rendering subtitles | ASS karaoke for Arabic/English |
| 7 | Rendering videos | FFmpeg single-pass + face crop |
| 8 | Validating outputs | ffprobe resolution/codec check |
| 9 | Generating metadata | LLM-based titles & hashtags |
| 10 | Producing outputs | Final summary JSON |

In [ ]:
from opusclip.config import PipelineConfig
from opusclip.provider_factory import ProviderFactory
from opusclip.exceptions import OpusClipError

# ── Configure ────────────────────────────────────────────────
config = PipelineConfig.from_env()
print(f"⚙   Output dir : {config.output_dir}")
print(f"⚙   LLM model  : {config.llm_model}")
print(f"⚙   Whisper    : {config.whisper_model} ({config.whisper_device})")
print(f"⚙   Encoder    : {config.encoder}")
print(f"⚙   Renderer   : {config.renderer_backend}")
print()

# ── Build pipeline ──────────────────────────────────────────
factory = ProviderFactory(config)
pipeline = factory.create_pipeline(VIDEO_PATH)

# ── Run ─────────────────────────────────────────────────────
print("▶  Starting pipeline …\n")
import time
t0 = time.time()

try:
    result = pipeline.run(VIDEO_PATH)
    elapsed = time.time() - t0
    print(f"\n✅  Pipeline finished in {elapsed:.1f}s")
except OpusClipError as e:
    elapsed = time.time() - t0
    print(f"\n❌  Pipeline failed after {elapsed:.1f}s: {e}")

---
## 5. Results

Preview the pipeline output: generated clips, metadata, and performance metrics.

In [ ]:
from opusclip.pipeline import PipelineResult

print(f"📊  Source duration : {result.duration:.1f}s ({result.duration/60:.1f} min)")
print(f"📊  Total clips    : {result.total_clips}")
print(f"📊  Successful     : {result.successful_clips}")
print(f"📊  Failed         : {result.failed_clips}")
print()

for clip in result.clips:
    status = "✅" if clip.success else "❌"
    print(f"  {status}  Clip #{clip.number}")
    if clip.video_path and clip.video_path.exists():
        size_mb = clip.video_path.stat().st_size / 1_048_576
        print(f"       Video : {clip.video_path.name} ({size_mb:.1f} MB)")
    if clip.thumbnail_path and clip.thumbnail_path.exists():
        print(f"       Thumb : {clip.thumbnail_path.name}")
    if clip.metadata:
        print(f"       Title : {clip.metadata.title}")
    if clip.error:
        print(f"       Error : {clip.error}")
    print()

# ── Pipeline metrics ────────────────────────────────────────
print("⏱  Pipeline Metrics:")
print(pipeline.metrics.report())

---
## 6. Download Results

Download individual clips or the entire output folder as a ZIP archive.

In [ ]:
import zipfile, shutil

OUTPUT_DIR = str(config.output_dir)

# ── ZIP the entire output ───────────────────────────────────
zip_path = "opusclip_results.zip"
if os.path.exists(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    print(f"📦  Creating {zip_path} …")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(OUTPUT_DIR):
            for fn in files:
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, OUTPUT_DIR))
    print(f"     ✓  {zip_path} created ({os.path.getsize(zip_path) / 1_048_576:.1f} MB)")
else:
    print("⚠  No output files found.")

# ── Download helper ─────────────────────────────────────────
try:
    from google.colab import files
    if os.path.exists(zip_path):
        print("\n⬇️  Click to download:")
        files.download(zip_path)
    for clip in result.clips:
        if clip.success and clip.video_path and clip.video_path.exists():
            files.download(str(clip.video_path))
except ImportError:
    print("\n📁  Output files are in:", os.path.abspath(OUTPUT_DIR))

---
## Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| `OPUSCLIP_API_KEY not set` | Missing API key | Set the key in Secrets or inline |
| `No module named 'opusclip'` | Package not installed | Restart runtime + re-run Cell 1 |
| `ffmpeg not found` | Missing binary | Re-run Cell 1 (install step) |
| `CUDA out of memory` | GPU OOM on Colab | Switch Whisper to `"small"` or `"medium"` |
| Pipeline slow | No GPU selected | **Runtime → Change runtime type → T4 GPU** |
| YouTube download fails | Region/age restriction | Try a different URL or use `"upload"` mode |
| Blank subtitles | Missing fonts | Re-run Cell 1 (font install step) |

---
*For more details, see the [OpusClip documentation](https://github.com/ABo-EsMaiL/OpusClip).*